# RetailPulse -- Baseline Prophet Model for Demand Forecasting

**Objective:** Build a baseline demand forecasting model using Facebook Prophet. Evaluate with train/test split, compute MAPE and RMSE, visualize forecast components, and generate 30-day ahead predictions.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load Data

In [3]:
df = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_ready.csv"), parse_dates=["ds"])
df = df.sort_values("ds").reset_index(drop=True)

print(f"Total records: {len(df)}")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Mean daily revenue: {df['y'].mean():,.2f}")
print(f"Std daily revenue: {df['y'].std():,.2f}")
df.tail()


Total records: 374
Date range: 2009-12-01 to 2010-12-09
Mean daily revenue: 23,524.69
Std daily revenue: 16,732.36


,ds,y
369,2010-12-05,31361.28
370,2010-12-06,31009.33
371,2010-12-07,53730.96
372,2010-12-08,39094.20
373,2010-12-09,38193.91


## Train / Test Split

Use the last 30 days as the test set. This simulates a real-world scenario where we train on historical data and evaluate on the most recent period.


In [4]:
TEST_DAYS = 30

split_date = df["ds"].max() - pd.Timedelta(days=TEST_DAYS)
train = df[df["ds"] <= split_date].copy()
test = df[df["ds"] > split_date].copy()

print(f"Train set: {len(train)} days ({train['ds'].min().date()} to {train['ds'].max().date()})")
print(f"Test set:  {len(test)} days ({test['ds'].min().date()} to {test['ds'].max().date()})")

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train["ds"], train["y"], color="#3498db", label="Train", linewidth=0.8)
ax.plot(test["ds"], test["y"], color="#e74c3c", label="Test", linewidth=1.5)
ax.axvline(split_date, color="black", linestyle="--", alpha=0.5, label="Split Point")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.set_title("Train / Test Split")
ax.legend()
fig.tight_layout()
save_fig(fig, "22_train_test_split.png")
plt.show()


Train set: 344 days (2009-12-01 to 2010-11-09)
Test set:  30 days (2010-11-10 to 2010-12-09)


Saved: 22_train_test_split.png


## Baseline Prophet Model

Start with default Prophet parameters to establish a performance baseline.


In [5]:
# Baseline model with default parameters
model_baseline = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=False,  # less than 1 year of data
)
model_baseline.fit(train)

# Predict on test period
future_test = model_baseline.make_future_dataframe(periods=TEST_DAYS)
forecast_baseline = model_baseline.predict(future_test)

# Extract test predictions
pred_baseline = forecast_baseline[forecast_baseline["ds"].isin(test["ds"])][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].reset_index(drop=True)
test_eval = test.reset_index(drop=True)

print(f"Predictions generated for {len(pred_baseline)} days")
pred_baseline.head()


10:28:25 - cmdstanpy - INFO - Chain [1] start processing


10:28:27 - cmdstanpy - INFO - Chain [1] done processing


Predictions generated for 30 days


,ds,yhat,yhat_lower,yhat_upper
0,2010-11-10,38265.372218,23618.355628,52422.189388
1,2010-11-11,45124.536131,31644.110153,59184.366746
2,2010-11-12,35665.020521,20597.320546,49992.119750
3,2010-11-13,12676.235501,-1705.058721,27643.545448
4,2010-11-14,30851.172933,16619.017693,45602.716220


In [6]:
def compute_metrics(actual, predicted, model_name=""):
    """Compute forecasting evaluation metrics."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    # Avoid division by zero for MAPE
    non_zero = actual != 0
    if non_zero.sum() > 0:
        mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
    else:
        mape = np.nan

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mse = mean_squared_error(actual, predicted)

    metrics = {
        "model": model_name,
        "MAPE (%)": round(mape, 2),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MSE": round(mse, 2),
    }

    if model_name:
        print(f"{model_name} Metrics:")
        print(f"  MAPE:  {mape:.2f}%")
        print(f"  MAE:   {mae:,.2f}")
        print(f"  RMSE:  {rmse:,.2f}")
    print()

    return metrics


In [7]:
baseline_metrics = compute_metrics(
    test_eval["y"].values, pred_baseline["yhat"].values, "Baseline Prophet")


Baseline Prophet Metrics:
  MAPE:  19.69%
  MAE:   9,652.33
  RMSE:  12,441.16



In [8]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(test_eval["ds"], test_eval["y"], color="#2c3e50", linewidth=1.5,
        label="Actual", marker="o", markersize=3)
ax.plot(pred_baseline["ds"], pred_baseline["yhat"], color="#e74c3c", linewidth=1.5,
        label="Predicted", marker="s", markersize=3)
ax.fill_between(pred_baseline["ds"], pred_baseline["yhat_lower"],
                pred_baseline["yhat_upper"], alpha=0.15, color="#e74c3c",
                label="Confidence Interval")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.set_title(f"Baseline Prophet -- Test Set (MAPE: {baseline_metrics['MAPE (%)']:.2f}%)")
ax.legend()
fig.tight_layout()
save_fig(fig, "23_baseline_prophet_forecast.png")
plt.show()


Saved: 23_baseline_prophet_forecast.png


## Tuned Prophet Model

Tune Prophet by adjusting:
- **changepoint_prior_scale**: flexibility of trend (higher = more flexible)
- **seasonality_prior_scale**: strength of seasonality
- **seasonality_mode**: additive vs multiplicative
- Add **monthly seasonality** and **country holidays**


In [9]:
# Grid search over key parameters
param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1, 0.5],
    "seasonality_prior_scale": [0.1, 1.0, 10.0],
    "seasonality_mode": ["additive", "multiplicative"],
}

best_mape = float("inf")
best_params = {}
results = []

total_combos = (len(param_grid["changepoint_prior_scale"])
                * len(param_grid["seasonality_prior_scale"])
                * len(param_grid["seasonality_mode"]))
print(f"Testing {total_combos} parameter combinations...")
print()

combo = 0
for cp_scale in param_grid["changepoint_prior_scale"]:
    for s_scale in param_grid["seasonality_prior_scale"]:
        for s_mode in param_grid["seasonality_mode"]:
            combo += 1
            m = Prophet(
                changepoint_prior_scale=cp_scale,
                seasonality_prior_scale=s_scale,
                seasonality_mode=s_mode,
                weekly_seasonality=True,
                daily_seasonality=False,
                yearly_seasonality=False,
            )
            m.add_seasonality(name="monthly", period=30.5, fourier_order=5)
            m.fit(train)

            future = m.make_future_dataframe(periods=TEST_DAYS)
            fc = m.predict(future)
            pred = fc[fc["ds"].isin(test["ds"])]["yhat"].values

            actual = test_eval["y"].values
            non_zero = actual != 0
            if non_zero.sum() > 0:
                mape = np.mean(np.abs((actual[non_zero] - pred[non_zero]) / actual[non_zero])) * 100
            else:
                mape = np.nan

            results.append({
                "changepoint_prior_scale": cp_scale,
                "seasonality_prior_scale": s_scale,
                "seasonality_mode": s_mode,
                "MAPE": round(mape, 2)
            })

            if mape < best_mape:
                best_mape = mape
                best_params = {
                    "changepoint_prior_scale": cp_scale,
                    "seasonality_prior_scale": s_scale,
                    "seasonality_mode": s_mode,
                }

results_df = pd.DataFrame(results).sort_values("MAPE")
print("Top 10 parameter combinations:")
print(results_df.head(10).to_string(index=False))
print()
print(f"Best parameters: {best_params}")
print(f"Best MAPE: {best_mape:.2f}%")


10:28:27 - cmdstanpy - INFO - Chain [1] start processing


10:28:27 - cmdstanpy - INFO - Chain [1] done processing


Testing 24 parameter combinations...



10:28:28 - cmdstanpy - INFO - Chain [1] start processing


10:28:28 - cmdstanpy - INFO - Chain [1] done processing


...

10:28:32 - cmdstanpy - INFO - Chain [1] start processing


10:28:33 - cmdstanpy - INFO - Chain [1] done processing


10:28:33 - cmdstanpy - INFO - Chain [1] start processing


10:28:33 - cmdstanpy - INFO - Chain [1] done processing


Top 10 parameter combinations:
 changepoint_prior_scale  seasonality_prior_scale seasonality_mode  MAPE
                    0.05                      0.1         additive 22.49
                    0.05                      1.0         additive 22.49
                    0.05                     10.0         additive 22.49
                    0.05                      0.1   multiplicative 23.80
                    0.05                      1.0   multiplicative 23.84
                    0.01                      0.1   multiplicative 23.88
                    0.10                     10.0         additive 23.97
                    0.10                      1.0         additive 23.99
                    0.10                      0.1         additive 24.03
                    0.05                     10.0   multiplicative 24.03

Best parameters: {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive'}
Best MAPE: 22.49%


In [10]:
# Fit the tuned model with best parameters
model_tuned = Prophet(
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"],
    seasonality_mode=best_params["seasonality_mode"],
    weekly_seasonality=True,
    daily_seasonality=False,
    yearly_seasonality=False,
)
model_tuned.add_seasonality(name="monthly", period=30.5, fourier_order=5)
model_tuned.add_country_holidays(country_name="GB")
model_tuned.fit(train)

future_test = model_tuned.make_future_dataframe(periods=TEST_DAYS)
forecast_tuned = model_tuned.predict(future_test)

pred_tuned = forecast_tuned[forecast_tuned["ds"].isin(test["ds"])][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].reset_index(drop=True)

tuned_metrics = compute_metrics(
    test_eval["y"].values, pred_tuned["yhat"].values, "Tuned Prophet")


10:28:33 - cmdstanpy - INFO - Chain [1] start processing


10:28:33 - cmdstanpy - INFO - Chain [1] done processing


Tuned Prophet Metrics:
  MAPE:  22.82%
  MAE:   10,825.61
  RMSE:  13,402.68



In [11]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(test_eval["ds"], test_eval["y"], color="#2c3e50", linewidth=1.5,
        label="Actual", marker="o", markersize=3)
ax.plot(pred_tuned["ds"], pred_tuned["yhat"], color="#27ae60", linewidth=1.5,
        label="Tuned Prophet", marker="s", markersize=3)
ax.fill_between(pred_tuned["ds"], pred_tuned["yhat_lower"],
                pred_tuned["yhat_upper"], alpha=0.15, color="#27ae60",
                label="Confidence Interval")
ax.plot(pred_baseline["ds"], pred_baseline["yhat"], color="#e74c3c",
        linewidth=1, linestyle="--", alpha=0.5, label="Baseline Prophet")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.set_title(f"Tuned Prophet -- Test Set (MAPE: {tuned_metrics['MAPE (%)']:.2f}%)")
ax.legend()
fig.tight_layout()
save_fig(fig, "24_tuned_prophet_forecast.png")
plt.show()


Saved: 24_tuned_prophet_forecast.png


## Forecast Components

In [12]:
# Prophet component plots
fig = model_tuned.plot_components(forecast_tuned)
fig.set_size_inches(14, 12)
fig.suptitle("Prophet Forecast Components (Tuned Model)", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "25_prophet_components.png")
plt.show()


Saved: 25_prophet_components.png


## 30-Day Future Forecast

Retrain the tuned model on the full dataset and generate a 30-day ahead forecast beyond the available data.


In [13]:
# Retrain on full dataset
model_final = Prophet(
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"],
    seasonality_mode=best_params["seasonality_mode"],
    weekly_seasonality=True,
    daily_seasonality=False,
    yearly_seasonality=False,
)
model_final.add_seasonality(name="monthly", period=30.5, fourier_order=5)
model_final.add_country_holidays(country_name="GB")
model_final.fit(df)

# Generate 30-day future forecast
future_30d = model_final.make_future_dataframe(periods=30)
forecast_30d = model_final.predict(future_30d)

# Extract future predictions only
future_only = forecast_30d[forecast_30d["ds"] > df["ds"].max()][
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].reset_index(drop=True)

print("30-Day Future Forecast:")
print(f"  Period: {future_only['ds'].min().date()} to {future_only['ds'].max().date()}")
print(f"  Avg predicted revenue: {future_only['yhat'].mean():,.2f}")
print(f"  Min predicted revenue: {future_only['yhat'].min():,.2f}")
print(f"  Max predicted revenue: {future_only['yhat'].max():,.2f}")
print()
future_only


10:28:35 - cmdstanpy - INFO - Chain [1] start processing


10:28:35 - cmdstanpy - INFO - Chain [1] done processing


30-Day Future Forecast:
  Period: 2010-12-10 to 2011-01-08
  Avg predicted revenue: 37,478.43
  Min predicted revenue: -8,824.78
  Max predicted revenue: 53,299.29



,ds,yhat,yhat_lower,yhat_upper
0,2010-12-10,42002.535812,28443.550513,55829.125929
1,2010-12-11,15553.304802,1821.143062,30468.885524
2,2010-12-12,33517.988686,19896.653687,47700.630542
3,2010-12-13,43393.636564,28626.422154,57463.819256
4,2010-12-14,47350.223068,32776.133630,61274.045894
5,2010-12-15,45783.757104,31025.135766,59470.275597
6,2010-12-16,52280.004201,38509.741303,65519.503314
7,2010-12-17,42588.801955,27433.222064,56303.758882
8,2010-12-18,16510.734375,2302.060579,30753.109337
9,2010-12-19,34872.722905,21709.301595,49164.911666


In [14]:
fig, ax = plt.subplots(figsize=(18, 6))

# Plot last 60 days of actual + 30 days forecast
recent = df.tail(60)
ax.plot(recent["ds"], recent["y"], color="#2c3e50", linewidth=1.2,
        label="Historical", marker="o", markersize=2)
ax.plot(future_only["ds"], future_only["yhat"], color="#e74c3c", linewidth=2,
        label="30-Day Forecast", marker="s", markersize=4)
ax.fill_between(future_only["ds"], future_only["yhat_lower"],
                future_only["yhat_upper"], alpha=0.2, color="#e74c3c",
                label="Confidence Interval")
ax.axvline(df["ds"].max(), color="black", linestyle="--", alpha=0.5,
           label="Forecast Start")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.set_title("30-Day Demand Forecast (Prophet)")
ax.legend()
fig.tight_layout()
save_fig(fig, "26_30day_forecast.png")
plt.show()


Saved: 26_30day_forecast.png


In [15]:
# Save the 30-day forecast
forecast_path = os.path.join(PROCESSED_DIR, "prophet_forecast_30d.csv")
future_only.to_csv(forecast_path, index=False)
print(f"Saved 30-day forecast: {forecast_path}")

# Save full forecast for dashboard use
full_forecast_path = os.path.join(PROCESSED_DIR, "prophet_forecast_full.csv")
forecast_30d[["ds", "yhat", "yhat_lower", "yhat_upper", "trend",
              "weekly", "monthly"]].to_csv(full_forecast_path, index=False)
print(f"Saved full forecast: {full_forecast_path}")


Saved 30-day forecast: ../data/processed/prophet_forecast_30d.csv
Saved full forecast: ../data/processed/prophet_forecast_full.csv


## Model Comparison

In [16]:
comparison = pd.DataFrame([baseline_metrics, tuned_metrics])
comparison = comparison.set_index("model")
print("MODEL COMPARISON")
print("=" * 55)
print(comparison.to_string())
print()

improvement = ((baseline_metrics["MAPE (%)"] - tuned_metrics["MAPE (%)"]) 
               / baseline_metrics["MAPE (%)"] * 100)
print(f"MAPE improvement from tuning: {improvement:.1f}%")


MODEL COMPARISON
                  MAPE (%)       MAE      RMSE           MSE
model                                                       
Baseline Prophet     19.69   9652.33  12441.16  1.547824e+08
Tuned Prophet        22.82  10825.61  13402.68  1.796319e+08

MAPE improvement from tuning: -15.9%


## Summary

In [17]:
print("PROPHET DEMAND FORECASTING COMPLETE")
print("=" * 55)
print()
print(f"Training data: {len(train)} days")
print(f"Test data: {len(test)} days")
print()
print("Model Performance (Test Set):")
print(f"  Baseline Prophet MAPE: {baseline_metrics['MAPE (%)']:.2f}%")
print(f"  Tuned Prophet MAPE:    {tuned_metrics['MAPE (%)']:.2f}%")
print()
print(f"Best Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print()
print("Exported files:")
print(f"  prophet_forecast_30d.csv   (30-day future predictions)")
print(f"  prophet_forecast_full.csv  (full historical + future forecast)")
print()
print("Next steps:")
print("  - LSTM model implementation with PyTorch Lightning")
print("  - Hybrid ensemble of Prophet + LSTM predictions")


PROPHET DEMAND FORECASTING COMPLETE

Training data: 344 days
Test data: 30 days

Model Performance (Test Set):
  Baseline Prophet MAPE: 19.69%
  Tuned Prophet MAPE:    22.82%

Best Parameters:
  changepoint_prior_scale: 0.05
  seasonality_prior_scale: 10.0
  seasonality_mode: additive

Exported files:
  prophet_forecast_30d.csv   (30-day future predictions)
  prophet_forecast_full.csv  (full historical + future forecast)

Next steps:
  - LSTM model implementation with PyTorch Lightning
  - Hybrid ensemble of Prophet + LSTM predictions
